# Day 6 Exam: Production & Frontiers

**Coverage:** Notebooks 14-17 (inference optimization, eval metrics, autoregressive video, distillation), with review of previous days.

---

## Exam Rules

- **Total time: ~2 hours** (warmups ~15 min, main problems ~90 min)
- **No documentation, no Google, no LLM assistance.** Closed-book.
- **Honor system.** If you can't remember something, leave a comment and move on.
- Write clean, typed, documented code — as if submitting for code review.
- Run all test/assertion cells to validate your work before finishing.
- If stuck on a warmup for >7 min, move on. Main problems matter more.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import math

---

## Warmup Section (~15 minutes total)

Quick recall problems. 5 minutes each. Don't overthink these.

### Warmup 1: Causal Mask (5 min)

Implement `make_causal_mask(seq_len) -> torch.Tensor`.

**Returns:** A boolean tensor of shape `(seq_len, seq_len)` where:
- `True` in the **upper triangle** (positions that should be masked / cannot attend)
- `False` on and below the diagonal (positions that can attend)

This is the standard causal (autoregressive) attention mask: position `i` can attend to positions `0..i` but not `i+1..seq_len-1`.

In [ ]:
# Warmup 1: Your implementation



In [ ]:
# Warmup 1: Tests
mask = make_causal_mask(5)
assert mask.shape == (5, 5), f"Expected (5, 5), got {mask.shape}"
assert mask.dtype == torch.bool, f"Expected bool, got {mask.dtype}"

# Diagonal should be False (can attend to self)
assert not mask.diagonal().any(), "Diagonal should be False (can attend to self)"

# Upper triangle should be True (cannot attend to future)
for i in range(5):
    for j in range(i + 1, 5):
        assert mask[i, j], f"Position ({i},{j}) should be True (masked)"

# Lower triangle should be False (can attend to past)
for i in range(5):
    for j in range(0, i):
        assert not mask[i, j], f"Position ({i},{j}) should be False (can attend)"

# Edge case: seq_len=1
mask1 = make_causal_mask(1)
assert mask1.shape == (1, 1) and not mask1[0, 0]

print("Warmup 1 passed.")

### Warmup 2: Frechet Distance (Simplified) (5 min)

Implement `frechet_distance(mu1, sigma1, mu2, sigma2) -> float` — the simplified 1D Frechet distance between two Gaussian distributions.

**Formula:** `FD = (mu1 - mu2)^2 + sigma1 + sigma2 - 2 * sqrt(sigma1 * sigma2)`

**Args:**
- `mu1, mu2`: float — means of the two distributions
- `sigma1, sigma2`: float — variances (not std devs) of the two distributions

**Returns:** float — the Frechet distance (scalar, non-negative)

In [ ]:
# Warmup 2: Your implementation



In [ ]:
# Warmup 2: Tests
# Identical distributions should give FD = 0
fd_same = frechet_distance(1.0, 2.0, 1.0, 2.0)
assert abs(fd_same) < 1e-6, f"Identical distributions should give FD=0, got {fd_same}"

# Non-negative
fd_diff = frechet_distance(0.0, 1.0, 3.0, 4.0)
assert fd_diff >= 0, f"FD should be non-negative, got {fd_diff}"

# Known value: mu1=0, sigma1=1, mu2=0, sigma2=4
# FD = 0 + 1 + 4 - 2*sqrt(4) = 5 - 4 = 1.0
fd_known = frechet_distance(0.0, 1.0, 0.0, 4.0)
assert abs(fd_known - 1.0) < 1e-6, f"Expected FD=1.0, got {fd_known}"

# Symmetric
fd_ab = frechet_distance(1.0, 2.0, 3.0, 5.0)
fd_ba = frechet_distance(3.0, 5.0, 1.0, 2.0)
assert abs(fd_ab - fd_ba) < 1e-6, "FD should be symmetric"

print("Warmup 2 passed.")

### Warmup 3: GPU Memory Estimator (5 min)

Implement `estimate_gpu_memory(param_count, dtype_bytes=2) -> dict` that estimates GPU memory usage for training a model.

**Memory components:**
- **Parameters:** `param_count * dtype_bytes` bytes
- **Gradients:** same as parameters (same dtype)
- **Adam optimizer states:** `param_count * 4 * 2` bytes (two fp32 buffers: first moment + second moment, 4 bytes each)

**Returns:** dict with keys:
- `"params_gb"`: parameter memory in GB
- `"grads_gb"`: gradient memory in GB
- `"optimizer_gb"`: optimizer state memory in GB
- `"total_gb"`: sum of all three

Use `1 GB = 1e9 bytes`.

In [ ]:
# Warmup 3: Your implementation



In [ ]:
# Warmup 3: Tests
# 1B params, fp16 (2 bytes)
result = estimate_gpu_memory(1_000_000_000, dtype_bytes=2)

assert isinstance(result, dict), "Should return a dict"
assert set(result.keys()) == {"params_gb", "grads_gb", "optimizer_gb", "total_gb"}

# 1B params * 2 bytes = 2GB
assert abs(result["params_gb"] - 2.0) < 0.01, f"Expected ~2.0 GB params, got {result['params_gb']}"
assert abs(result["grads_gb"] - 2.0) < 0.01, f"Expected ~2.0 GB grads, got {result['grads_gb']}"

# Adam states: 1B * 4 bytes * 2 states = 8GB
assert abs(result["optimizer_gb"] - 8.0) < 0.01, f"Expected ~8.0 GB optimizer, got {result['optimizer_gb']}"

# Total = 2 + 2 + 8 = 12
assert abs(result["total_gb"] - 12.0) < 0.01, f"Expected ~12.0 GB total, got {result['total_gb']}"

# fp32 case: 1B params * 4 bytes = 4GB params, 4GB grads, 8GB optimizer = 16GB
result_fp32 = estimate_gpu_memory(1_000_000_000, dtype_bytes=4)
assert abs(result_fp32["total_gb"] - 16.0) < 0.01

print("Warmup 3 passed.")

---

## Main Section (~90 minutes total)

Substantial implementation problems. 30 minutes each. Write clean, tested code.

### Main Problem 1: Mini Eval Pipeline (30 min)

Implement an `EvalPipeline` class that computes evaluation metrics on generated images.

**Interface:**
```python
class EvalPipeline:
    def __init__(self, metrics: list[str]) -> None: ...
    def evaluate(
        self,
        predictions: list[torch.Tensor],
        references: list[torch.Tensor],
        prompts: list[str],
    ) -> dict: ...
```

**Supported metrics:**

1. **`"psnr"`** — Peak Signal-to-Noise Ratio between each prediction and reference pair:
   - `MSE = mean((pred - ref)^2)`
   - `PSNR = 10 * log10(MAX_VAL^2 / MSE)` where `MAX_VAL = 1.0` (assume images in [0, 1])
   - If MSE is 0, return PSNR = 100.0 (perfect match cap)

2. **`"diversity"`** — Mean pairwise L2 distance across all predictions (not per-pair with reference):
   - Flatten each prediction, compute L2 distance between all pairs `(i, j)` where `i < j`
   - Return the mean of all pairwise distances

**Return format:** dict with structure:
```python
{
    "psnr": {"mean": float, "std": float},
    "diversity": {"mean": float, "std": float},  # std of pairwise distances
}
```
Only include metrics that were requested in `__init__`.

In [ ]:
# Main Problem 1: Your implementation



In [ ]:
# Main Problem 1: Tests
torch.manual_seed(42)

pipeline = EvalPipeline(metrics=["psnr", "diversity"])

# Test 1: Identical images should give high PSNR
img = torch.rand(3, 32, 32)
preds_identical = [img.clone() for _ in range(5)]
refs = [img.clone() for _ in range(5)]
prompts = ["test"] * 5

result_identical = pipeline.evaluate(preds_identical, refs, prompts)
assert "psnr" in result_identical, "Should have 'psnr' in result"
assert result_identical["psnr"]["mean"] >= 90.0, f"Identical images should give very high PSNR, got {result_identical['psnr']['mean']}"

# Test 2: Diverse images should give higher diversity score than identical images
preds_diverse = [torch.rand(3, 32, 32) for _ in range(5)]
result_diverse = pipeline.evaluate(preds_diverse, refs, prompts)

result_same = pipeline.evaluate(preds_identical, refs, prompts)
assert result_diverse["diversity"]["mean"] > result_same["diversity"]["mean"], \
    "Diverse images should have higher diversity score"

# Test 3: Report has correct keys
for metric in ["psnr", "diversity"]:
    assert metric in result_diverse, f"Missing metric: {metric}"
    assert "mean" in result_diverse[metric], f"Missing 'mean' in {metric}"
    assert "std" in result_diverse[metric], f"Missing 'std' in {metric}"

# Test 4: Only requested metrics
pipeline_psnr_only = EvalPipeline(metrics=["psnr"])
result_psnr_only = pipeline_psnr_only.evaluate(preds_diverse, refs, prompts)
assert "psnr" in result_psnr_only
assert "diversity" not in result_psnr_only

print(f"Identical PSNR: {result_identical['psnr']['mean']:.1f} dB")
print(f"Diverse diversity: {result_diverse['diversity']['mean']:.4f}")
print(f"Identical diversity: {result_same['diversity']['mean']:.6f}")
print("Main Problem 1 passed.")

### Main Problem 2: Knowledge Distillation Training Loop (30 min)

Implement a `DistillationTrainer` class that trains a student model to match both ground-truth targets and a frozen teacher model's outputs.

**Interface:**
```python
class DistillationTrainer:
    def __init__(
        self,
        student: nn.Module,
        teacher: nn.Module,
        optimizer: torch.optim.Optimizer,
        alpha: float = 0.5,
    ) -> None: ...
    
    def train_step(self, x: torch.Tensor, t: torch.Tensor) -> dict: ...
```

**`train_step` behavior:**
1. Generate ground-truth noise: `noise = torch.randn_like(x)`
2. Compute `student_pred = student(x, t)` and `teacher_pred = teacher(x, t).detach()`
3. `student_loss = MSE(student_pred, noise)`
4. `distill_loss = MSE(student_pred, teacher_pred)`
5. `total_loss = alpha * student_loss + (1 - alpha) * distill_loss`
6. Backward + optimizer step (zero_grad, backward, step)
7. Return `{"loss": total_loss.item(), "student_loss": student_loss.item(), "distill_loss": distill_loss.item()}`

**Notes:**
- Teacher should be set to `eval()` and its parameters should never require gradients
- The noise target in `student_loss` is freshly sampled each step (simulating the diffusion training objective)

In [ ]:
# Main Problem 2: Your implementation



In [ ]:
# Main Problem 2: Tests
torch.manual_seed(42)

# Simple models for testing
class TinyModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 3, 3, padding=1),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        return self.net(x)


student = TinyModel()
teacher = TinyModel()
optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

trainer = DistillationTrainer(student, teacher, optimizer, alpha=0.5)

# Record teacher params before training
teacher_params_before = {n: p.clone() for n, p in teacher.named_parameters()}
student_params_before = {n: p.clone() for n, p in student.named_parameters()}

# Train for 5 steps and verify loss decreases
x = torch.randn(4, 3, 16, 16)
t = torch.randint(0, 100, (4,))
losses = []
for _ in range(5):
    result = trainer.train_step(x, t)
    losses.append(result["loss"])
    assert "loss" in result and "student_loss" in result and "distill_loss" in result, \
        f"Missing keys in result: {result.keys()}"

# Loss should generally decrease (allow some noise — just check last < first)
assert losses[-1] < losses[0], f"Loss should decrease: {losses[0]:.4f} -> {losses[-1]:.4f}"

# Teacher params should NOT change
for name, p in teacher.named_parameters():
    assert torch.equal(p, teacher_params_before[name]), f"Teacher param {name} changed!"

# Student params SHOULD change
student_changed = any(
    not torch.equal(p, student_params_before[n])
    for n, p in student.named_parameters()
)
assert student_changed, "Student params should have changed after training"

print(f"Losses over 5 steps: {[f'{l:.4f}' for l in losses]}")
print("Main Problem 2 passed.")

### Main Problem 3: Data Pipeline Estimator (30 min) — Review Problem

Implement a `DataPipelineEstimator` that estimates compute requirements for a video processing pipeline.

**Interface:**
```python
class DataPipelineEstimator:
    def estimate(
        self,
        n_videos: int,
        n_gpus: int,
        steps_per_video: int,
    ) -> dict: ...
```

**Pipeline stages and throughput (per GPU or per CPU core):**

| Stage | Hardware | Throughput (clips/sec) |
|---|---|---|
| `scene_detect` | CPU | 10 |
| `quality_filter` | CPU | 100 |
| `caption` | GPU | 50 |
| `embed` | GPU | 200 |
| `store` | CPU | 500 |

**Assumptions:**
- Each video produces `steps_per_video` clips that flow through all stages
- Total clips = `n_videos * steps_per_video`
- CPU stages run on a single core (not parallelized across GPUs)
- GPU stages are parallelized across `n_gpus`
- The pipeline is bottlenecked by the **slowest stage** (determines wall clock time)

**Returns:** dict with:
- `"total_clips"`: int — total clips to process
- `"stage_hours"`: dict mapping stage name to hours required for that stage
- `"bottleneck"`: str — name of the slowest stage
- `"total_gpu_hours"`: float — sum of (GPU stage hours * n_gpus) — total GPU-hours consumed
- `"wall_clock_hours"`: float — time for the bottleneck stage (pipeline limited by slowest)
- `"storage_tb"`: float — estimate at 50 MB per clip, converted to TB (1 TB = 1e6 MB)

In [ ]:
# Main Problem 3: Your implementation



In [ ]:
# Main Problem 3: Tests
estimator = DataPipelineEstimator()

# 1M videos, 8 GPUs, 10 clips per video
result = estimator.estimate(n_videos=1_000_000, n_gpus=8, steps_per_video=10)

assert isinstance(result, dict), "Should return a dict"
assert result["total_clips"] == 10_000_000, f"Expected 10M clips, got {result['total_clips']}"

# GPU hours should be positive
assert result["total_gpu_hours"] > 0, "GPU hours should be positive"

# Wall clock hours should be positive
assert result["wall_clock_hours"] > 0, "Wall clock hours should be positive"

# Storage: 10M clips * 50MB = 500 TB
assert abs(result["storage_tb"] - 500.0) < 1.0, f"Expected ~500 TB, got {result['storage_tb']}"

# Bottleneck should be scene_detect (10 clips/sec on CPU = 10M/10/3600 = 277.8 hours)
# vs quality_filter: 10M/100/3600 = 27.8h, caption: 10M/50/8/3600 = 6.94h, etc.
assert result["bottleneck"] == "scene_detect", f"Expected bottleneck='scene_detect', got '{result['bottleneck']}'"

# stage_hours should have all 5 stages
expected_stages = {"scene_detect", "quality_filter", "caption", "embed", "store"}
assert set(result["stage_hours"].keys()) == expected_stages, f"Missing stages: {expected_stages - set(result['stage_hours'].keys())}"

print(f"Total clips: {result['total_clips']:,}")
print(f"Bottleneck: {result['bottleneck']}")
print(f"Wall clock hours: {result['wall_clock_hours']:.1f}")
print(f"Total GPU hours: {result['total_gpu_hours']:.1f}")
print(f"Storage: {result['storage_tb']:.1f} TB")
print("Stage hours:")
for stage, hours in sorted(result["stage_hours"].items(), key=lambda x: -x[1]):
    print(f"  {stage}: {hours:.1f} h")
print("Main Problem 3 passed.")